# Dependency install

In [ ]:
!pip install -q docling "docling[asr]" semantic-chunking pymupdf requests pillow transformers python-docx python-pptx python-magic

In [ ]:
!pip install -q langchain-huggingface langchain-core langchain torch urlpolice

# File Ingestion

In [ ]:

from docling.document_converter import DocumentConverter, AudioFormatOption
from docling.datamodel import asr_model_specs
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import AsrPipelineOptions
from docling.pipeline.asr_pipeline import AsrPipeline
from semantic_chunking import SemanticChunker
import requests
from io import BytesIO
import fitz
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from docx import Document
from pptx import Presentation
import magic
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from urlpolice import URLPolice

# function to convert in embeddings and store in vector store
def addInVectorStore(chunks):

  docs = []
  for i in range(len(chunks)):
    docs.append(Document(page_content=chunks[i], metadata={"id" : i}))

  vector_store.add_documents(docs)

# function for generating captions for image
def imageCaptioning(link):

  resp = requests.get(link)
  file_stream = BytesIO(resp.content)
  mime = magic.from_buffer(file_stream.getvalue(), mime=True)

  images = []

  if mime == "application/pdf":
    docs = fitz.open(stream=file_stream, filetype="pdf")

    for page in docs:
      for img in page.get_images(full=True):
        xref = img[0]
        base_image = docs.extract_image(xref)
        image_bytes = base_image["image"]
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        images.append(image)

  elif mime == "application/vnd.openxmlformats-officedocument.wordprocessingml.document":
    doc = Document(file_stream)

    for rel in doc.part.rels.values():
      if "image" in rel.target_ref:
        image_bytes = rel.target_part.blob
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        images.append(image)

  elif mime == "application/vnd.ms-powerpoint":
    prs = Presentation(file_stream)

    for slide in prs.slides:
      for shape in slide.shapes:
        if shape.shape_type == 13: # picture
          image_bytes = shape.image.blob
          image = Image.open(BytesIO(image_bytes)).convert("RGB")
          images.append(image)

  processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
  model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

  captions = []

  for img in images:
    inputs = processor(img, return_tensors="pt")
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    captions.append(caption)

  return captions

# function for semantic chunking of texts
def semanticChunking(text):
  chunker = SemanticChunker(model_name='all-MiniLM-L6-v2', max_chunk_size=1500, similarity_threshold=0.7)
  chunks = chunker.semantic_chunk(text)

  print(f"Total chunk length : {len(chunks)}")
  print("\nSemantic Chunks:\n")
  for chunk in chunks:
    print(f"{chunk}\n")

  return chunks

# function for document extraction
def extractDocument(link):
  pdfconverter = DocumentConverter()
  doc = pdfconverter.convert(link).document
  content = doc.export_to_markdown()
  print(content)
  captions = imageCaptioning(link)

  if len(captions) > 0:
    content += "\n\n IMAGE DESCRIPTIONS : \n\n"
    for text in captions:
      content += f"{text}\n"

  chunks = semanticChunking(content)
  addInVectorStore(chunks)

# function for audio & video extraction
def mediaExtraction(link):
  pipeline_options = AsrPipelineOptions()
  pipeline_options.asr_options = asr_model_specs.WHISPER_TURBO

  mediaconverter = DocumentConverter(
       format_options = {
          InputFormat.AUDIO: AudioFormatOption(
             pipeline_cls = AsrPipeline,
             pipeline_options = pipeline_options
         )
      }
  )

  result = mediaconverter.convert(link).document
  content = result.export_to_markdown()

  chunks = semanticChunking(content)
  addInVectorStore(chunks)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = InMemoryVectorStore(embedding=embedding_model)
police = URLPolice()

link = input("Enter a link : ")

if "google.com" in link:
  print("URL not allowed")
else:
  validate = police.validate(link)
  if not validate.is_valid:
    print("URL not secure")
  else:
    resp = requests.get(link)
    file_stream = BytesIO(resp.content)
    mime = magic.from_buffer(file_stream.getvalue(), mime=True)

    if mime == "audio/mpeg" or mime == "video/mp4":
      mediaExtraction(link)
    else:
      extractDocument(link)

# Retrieval

In [ ]:

import torch
from transformers import pipeline

question = input("Ask any question based on your data : ")
allDocs = vector_store.similarity_search(query, k=10)

context = ""

for doc in allDocs:
  context += f"{doc.page_content}\n"

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")

query = f'''
Try to summarize the context and give answer of the query from it.
If any information is not available on the context then you can say that
you dont have relevant context for this question.

QUERY: {question}
CONTEXT: {context}
'''

messages = [
    {
        "role": "system",
        "content": '''You are a friendly chatbot who always responds based on the context provided.
         And if you dont find any matched context for the question then simply say that you dont have relevant context for this question.
         And send the response as plain text without any markdown formatting
         and no need to respond back the entire query, context, prompt. Simply respond with the answer.''',
    },
    {
        "role": "user",
        "content": f"{query}"
    },
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95, return_full_text=False)

print(outputs[0]["generated_text"])

# URL check

In [ ]:
from urlpolice import URLPolice

police = URLPolice()

result = police.validate("https://invalid-domain.com/storage/v1/object/public/temp/Post%20PHD%20P2.pdf")

if not result.is_valid:
  print("URL is not valid")
else:
  print("Valid URL")